# Financial Complaint Classification & Emerging Issue Detection

**Notebook revision v3:** uncapped training, identical holdouts and a preserved capped-run comparison. The restart-safe v2 setup is retained.

An end-to-end Python, SQL and NLP project using an official historical CFPB archive.

**Business question:** Can an analyst classify written complaints into financial product categories and identify unusual changes in reported issues?

## Run in this order

1. Upload this notebook to Google Colab and connect to a **CPU** runtime. A GPU is not used by these models.
2. Keep `PROFILE = "uncapped"` and run all cells. If setup installs packages, it deliberately stops with **ONE SESSION RESTART IS REQUIRED**. Choose **Runtime > Restart session**, then run all cells again from the top. Do not delete the runtime.
3. The notebook uses all **101,935 eligible training complaints**, with **9,411 validation** and **15,390 test complaints**. It verifies that the validation/test membership matches your earlier capped run before training.
4. Run the final download cell to get **Financial_Complaint_Results_uncapped.zip**. Save this executed notebook with its outputs. Your earlier `Financial_Complaint_Results_full.zip` remains the capped reference.
5. Return the uncapped results ZIP for verification and dashboard integration. Selection between capped and uncapped models uses **validation macro-F1**, not test scores.

No Kaggle account, API key, source-code upload or manual dataset upload is required. The original archive is about 72 MiB compressed. The database and processing require additional local runtime storage. Runtime resources and session duration depend on Colab availability.

**Evidence boundary:** the code is checked locally using the smoke profile and real-data partition checks. Your earlier capped Colab run is the recorded reference. The uncapped Colab training run has not been performed yet.

## 1. Settings

The default is `uncapped`; your smoke and capped runs have already completed. Keep the same runtime when possible to reuse the verified download and database. The original `full` profile remains capped at 15,000 per category; `smoke` remains a small optional pipeline check.
Optional Drive saving requires your Google authorization and copies only the final results ZIP to Drive.

In [ ]:
PROFILE = "uncapped"  # Use all eligible training complaints; preserves the earlier capped full profile.
SAVE_RESULTS_TO_DRIVE = False  # Optional; requires your authorization.

from pathlib import Path
import os, sys, json, shutil, subprocess, importlib.util
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
PROFILE = os.environ.get("FCP_PROFILE", PROFILE)  # Local QA can override the profile without editing this notebook.
assert PROFILE in ("smoke", "full", "uncapped")
assert sys.version_info >= (3,12), "Use a Python 3.12 or 3.13 runtime."
ROOT = Path(os.environ.get("FCP_PROJECT_ROOT", "/content/FinancialComplaintIntelligence" if IN_COLAB else "FinancialComplaintIntelligence")).resolve()
ROOT.mkdir(parents=True, exist_ok=True)
EXECUTION_ENVIRONMENT = "colab_cpu" if IN_COLAB else "local_notebook_qa"
print({"profile": PROFILE, "environment": EXECUTION_ENVIRONMENT, "folder": str(ROOT)})

## 2. Install the reproducible training environment

Version pins keep the exported model compatible with the local application. Colab can preload NumPy before your first visible import. Continuing immediately after replacing it can mix old binary modules with new Python files, producing errors such as `cannot import name '_slice'`.

**When packages change, this cell deliberately stops before importing the project. Choose Runtime > Restart session, then run all cells again from the top.** Installed files remain available, so the installation will not repeat. Rerunning the failing import cell without restarting is blocked. A fresh-interpreter check also verifies that the installed package files work independently of the old kernel.

The setup includes compatible Bottleneck and NumExpr versions to address pandas' optional-dependency warnings.

In [ ]:
RUNTIME_SETUP_SOURCE = '"""Install the notebook environment without using stale in-memory binary modules."""\nfrom __future__ import annotations\n\nimport importlib\nfrom importlib import metadata\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nMODULES={"scikit-learn":"sklearn","bottleneck":"bottleneck"}\nMARKER=".runtime_restart_required.json"\nRESTART_MESSAGE=(\n    "SETUP COMPLETE: ONE SESSION RESTART IS REQUIRED. "\n    "In Colab choose Runtime > Restart session (not Disconnect and delete runtime). "\n    "Then run all cells again from Settings. Installed packages and runtime files are retained. "\n    "Do not rerun the failing import cell in this same session."\n)\n\n\ndef distribution_versions(required):\n    found={}\n    for package in required:\n        try:\n            found[package]=metadata.version(package)\n        except metadata.PackageNotFoundError:\n            found[package]=None\n    return found\n\n\ndef install_packages(specs):\n    subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--upgrade",*specs])\n\n\ndef import_versions(required):\n    return {package:importlib.import_module(MODULES.get(package,package)).__version__ for package in required}\n\n\ndef probe_fresh_interpreter(required):\n    # A child process tests the installed files without reusing this kernel\'s NumPy.\n    script=("import importlib,json; required="+repr(required)+"; modules="+repr(MODULES)+"; "\n            "actual={p:importlib.import_module(modules.get(p,p)).__version__ for p in required}; "\n            "assert actual==required,(actual,required); "\n            "import numpy as np; assert np.char.lower(np.array([\'CHECK\']))[0]==\'check\'; "\n            "print(json.dumps(actual))")\n    process=subprocess.run([sys.executable,"-c",script],capture_output=True,text=True,timeout=180)\n    if process.returncode:\n        raise RuntimeError("The installed environment also fails in a fresh Python process. Installation needs repair before training.\\n"+process.stderr[-5000:])\n    return json.loads(process.stdout.strip().splitlines()[-1])\n\n\ndef mark_restart(root,reason):\n    marker=Path(root)/MARKER\n    marker.parent.mkdir(parents=True,exist_ok=True)\n    marker.write_text(json.dumps({"pid":os.getpid(),"reason":reason}),encoding="utf-8")\n    raise RuntimeError(RESTART_MESSAGE)\n\n\ndef assert_no_pending_restart(root):\n    marker=Path(root)/MARKER\n    if marker.exists():\n        pending=json.loads(marker.read_text(encoding="utf-8"))\n        if pending["pid"]==os.getpid():\n            raise RuntimeError(RESTART_MESSAGE)\n\n\ndef ensure_runtime(required,root,in_colab=False):\n    root=Path(root)\n    assert_no_pending_restart(root)\n    installed=distribution_versions(required)\n    changes=[f"{p}=={wanted}" for p,wanted in required.items() if installed[p]!=wanted]\n    loaded=[p for p in required if MODULES.get(p,p) in sys.modules]\n    if changes:\n        install_packages(changes)\n        probe_fresh_interpreter(required)\n        # Colab can preload numeric packages before a notebook\'s first import.\n        if in_colab or loaded:\n            mark_restart(root,"Dependencies were installed or changed in this kernel.")\n    try:\n        active=import_versions(required)\n    except (ImportError,AttributeError,ValueError) as exc:\n        probe_fresh_interpreter(required)\n        try:\n            mark_restart(root,"Installed files pass a fresh-process check but current imports fail.")\n        except RuntimeError as restart:\n            raise restart from exc\n    if active!=required:\n        probe_fresh_interpreter(required)\n        mark_restart(root,"Loaded package versions differ from the installed requirements.")\n    (root/MARKER).unlink(missing_ok=True)\n    print("Environment checks passed:",active)\n    return active\n'
(ROOT / "runtime_setup.py").write_text(RUNTIME_SETUP_SOURCE, encoding="utf-8")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import runtime_setup
import importlib
importlib.reload(runtime_setup)
REQUIRED = {"numpy":"2.5.4", "pandas":"3.1.0", "scipy":"1.18.1", "scikit-learn":"1.9.1", "joblib":"1.6.0", "matplotlib":"3.11.2", "bottleneck":"1.6.0", "numexpr":"2.14.2"}
runtime_setup.ensure_runtime(REQUIRED, ROOT, in_colab=IN_COLAB)

## 3. Install the transparent pipeline source

The cell below contains the complete project module, not a dependency on a private repository. It writes `complaint_project.py` into the runtime so the saved preprocessing function can also be imported by the local application.
You can inspect the file in Colab's Files panel.

In [ ]:
PIPELINE_SOURCE = '"""Reproducible CFPB analytics and text classification pipeline (CPU only)."""\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import Counter\nimport csv\nfrom contextlib import closing\nfrom datetime import datetime, timezone\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport sqlite3\nimport subprocess\nimport sys\nimport time\nimport urllib.request\nimport zipfile\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, balanced_accuracy_score\nfrom sklearn.naive_bayes import ComplementNB\nfrom sklearn.pipeline import Pipeline\n\nVERSION = "1.1.0"\nDATA_VERSION = "1.0.0"  # Cleaning/database schema are unchanged by the new profile.\nSOURCE_URL = "https://files.consumerfinance.gov/f/documents/CCDB_Export_9_January_2025_through_February_2025.zip"\nSOURCE_PAGE = "https://www.consumerfinance.gov/foia-requests/foia-electronic-reading-room/cfpb-consumer-complaint-database-narratives-archive/"\nSOURCE_SHA256 = "11c18bb0973eed8f5e4f66e2979346af0671a3aefcb5a6b9eec73009d53cff64"\nSEED = 42\nPROFILES = {\n    "smoke": {"train_cap_per_class": 250, "eval_cap_per_class": 80, "max_features": 18000},\n    "full": {"train_cap_per_class": 15000, "eval_cap_per_class": None, "max_features": 65000},\n    "uncapped": {"train_cap_per_class": None, "eval_cap_per_class": None, "max_features": 65000},\n}\n\n\ndef write_json(path: Path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n", encoding="utf-8")\n\n\ndef file_sha(path: Path) -> str:\n    with path.open("rb") as f:\n        return hashlib.file_digest(f, "sha256").hexdigest()\n\n\ndef normalize_text(value: str) -> str:\n    """Remove redaction artifacts and variable identifiers, retaining complaint words."""\n    value = str(value).lower()\n    value = re.sub(r"https?://\\S+|www\\.\\S+|\\b[^\\s@]+@[^\\s@]+\\.[^\\s@]+\\b", " ", value)\n    value = re.sub(r"\\bx{2,}\\b|\\d+", " ", value)\n    value = re.sub(r"[^a-z\\s\']", " ", value)\n    return " ".join(value.split())\n\n\ndef download_archive(root: Path) -> Path:\n    target = root / "data/raw/cfpb_2025_01_02.zip"\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.exists() and file_sha(target) == SOURCE_SHA256:\n        print("Verified cached official archive.", flush=True)\n        return target\n    temporary = target.with_suffix(".download")\n    request = urllib.request.Request(SOURCE_URL, headers={"User-Agent": "Mozilla/5.0"})\n    try:\n        with urllib.request.urlopen(request, timeout=90) as response, temporary.open("wb") as output:\n            shutil.copyfileobj(response, output, length=1024 * 1024)\n    except Exception as exc:\n        # The official CDN can reject Python clients. Curl follows the same public URL.\n        curl = shutil.which("curl.exe") or shutil.which("curl")\n        if not curl:\n            raise RuntimeError("Download failed. Upload the official archive to data/raw/cfpb_2025_01_02.zip and rerun.") from exc\n        subprocess.run([curl, "--fail", "--location", "--retry", "2", "--max-time", "300", "--output", str(temporary), SOURCE_URL], check=True)\n    if file_sha(temporary) != SOURCE_SHA256:\n        temporary.unlink(missing_ok=True)\n        raise ValueError("Source checksum changed. Inspect the new official release before using it.")\n    temporary.replace(target)\n    return target\n\n\ndef build_database(root: Path, archive_path: Path) -> dict:\n    """Stream all records into SQLite; do not load the full raw CSV into RAM."""\n    db_path = root / "data/complaints.sqlite"\n    manifest_path = root / "reports/data_manifest.json"\n    if db_path.exists() and manifest_path.exists():\n        existing = json.loads(manifest_path.read_text(encoding="utf-8"))\n        if existing.get("data_pipeline_version",existing.get("pipeline_version")) == DATA_VERSION and existing.get("source_sha256") == file_sha(archive_path):\n            with closing(sqlite3.connect(db_path)) as con:\n                if con.execute("PRAGMA quick_check").fetchone()[0] == "ok":\n                    print("Reusing verified SQLite database.", flush=True)\n                    return existing\n    db_path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = db_path.with_suffix(".building.sqlite")\n    temporary.unlink(missing_ok=True)\n    counts = Counter()\n    csv.field_size_limit(20_000_000)\n    with closing(sqlite3.connect(temporary)) as con:\n        con.execute("""CREATE TABLE complaints (\n            complaint_id TEXT PRIMARY KEY, date_received TEXT NOT NULL,\n            product TEXT NOT NULL, issue TEXT NOT NULL, company TEXT NOT NULL,\n            state TEXT NOT NULL, company_response TEXT NOT NULL, timely_response TEXT NOT NULL,\n            has_narrative INTEGER NOT NULL, clean_text TEXT, text_hash TEXT)""")\n        insert = "INSERT OR IGNORE INTO complaints VALUES (?,?,?,?,?,?,?,?,?,?,?)"\n        batch = []\n        with zipfile.ZipFile(archive_path) as archive:\n            for member in archive.namelist():\n                if not member.lower().endswith(".csv"):\n                    continue\n                with archive.open(member) as f:\n                    reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8-sig", newline=""))\n                    required = {"Complaint ID", "Date received", "Product", "Consumer complaint narrative", "Issue"}\n                    if not required.issubset(reader.fieldnames or []):\n                        raise ValueError("Unexpected archive schema.")\n                    for row in reader:\n                        counts["source_rows"] += 1\n                        received = row["Date received"].strip()\n                        if not re.fullmatch(r"2025-(01|02)-\\d\\d", received):\n                            raise ValueError(f"Unexpected source date: {received!r}")\n                        narrative = row["Consumer complaint narrative"].strip()\n                        counts["with_narrative"] += bool(narrative)\n                        text = normalize_text(narrative) if narrative else ""\n                        eligible = len(text.split()) >= 12\n                        counts["narratives_too_short"] += bool(narrative) and not eligible\n                        digest = hashlib.sha256(text.encode()).hexdigest() if eligible else None\n                        batch.append((row["Complaint ID"], received, row["Product"].strip(), row["Issue"].strip(),\n                                      row.get("Company", "").strip(), row.get("State", "").strip(),\n                                      row.get("Company response to consumer", "").strip(), row.get("Timely response?", "").strip(),\n                                      int(bool(narrative)), text if eligible else None, digest))\n                        if len(batch) >= 10000:\n                            con.executemany(insert, batch)\n                            con.commit()\n                            batch.clear()\n                            if counts["source_rows"] % 100000 == 0:\n                                print(f"Loaded {counts[\'source_rows\']:,} source rows...", flush=True)\n        if batch:\n            con.executemany(insert, batch)\n        con.execute("CREATE INDEX idx_date_product ON complaints(date_received,product)")\n        con.execute("CREATE INDEX idx_text_hash ON complaints(text_hash,date_received,complaint_id)")\n        con.commit()\n        counts["database_rows"] = con.execute("SELECT COUNT(*) FROM complaints").fetchone()[0]\n    temporary.replace(db_path)\n    manifest = {\n        "pipeline_version": VERSION, "data_pipeline_version": DATA_VERSION, "source_url": SOURCE_URL, "source_page": SOURCE_PAGE,\n        "source_sha256": file_sha(archive_path), "source_bytes": archive_path.stat().st_size,\n        "date_min": "2025-01-01", "date_max": "2025-02-28", **dict(counts),\n        "duplicate_ids_removed": counts["source_rows"] - counts["database_rows"],\n        "text_filter": "At least 12 normalized words; URLs, emails, digits and redaction markers removed.",\n        "analytics_population": "All archived records, including records without narrative text.",\n        "limitations": "Historical self-selected, unverified complaints; counts do not measure company failure rates or prove misconduct.",\n    }\n    write_json(manifest_path, manifest)\n    return manifest\n\n\ndef export_analytics(root: Path) -> dict:\n    output = root / "reports/analytics"\n    output.mkdir(parents=True, exist_ok=True)\n    queries = {\n        "daily": "SELECT date_received AS date, product, COUNT(*) AS complaints, SUM(has_narrative) AS narratives FROM complaints GROUP BY 1,2 ORDER BY 1,2",\n        "products": "SELECT product, COUNT(*) AS complaints, SUM(has_narrative) AS narratives, SUM(CASE WHEN timely_response=\'Yes\' THEN 1 ELSE 0 END) AS timely, SUM(CASE WHEN timely_response IN (\'Yes\',\'No\') THEN 1 ELSE 0 END) AS timeliness_known FROM complaints GROUP BY 1 ORDER BY complaints DESC",\n        "issues": "SELECT product, issue, COUNT(*) AS complaints FROM complaints GROUP BY 1,2 ORDER BY complaints DESC",\n        "responses": "SELECT company_response, COUNT(*) AS complaints FROM complaints GROUP BY 1 ORDER BY complaints DESC",\n        "monthly": "SELECT substr(date_received,1,7) AS month, product, COUNT(*) AS complaints FROM complaints GROUP BY 1,2 ORDER BY 1,2",\n        "states": "SELECT state, COUNT(*) AS complaints FROM complaints WHERE state<>\'\' GROUP BY 1 ORDER BY complaints DESC",\n    }\n    analytics = {}\n    with closing(sqlite3.connect(root / "data/complaints.sqlite")) as con:\n        for name, query in queries.items():\n            frame = pd.read_sql_query(query, con)\n            frame.to_csv(output / f"{name}.csv", index=False)\n            analytics[name] = frame.to_dict(orient="records")\n        weekly = pd.read_sql_query("""SELECT date(date_received, printf(\'-%d days\',\n           (CAST(strftime(\'%w\',date_received) AS INTEGER)+6)%7)) AS week,\n           product,issue,COUNT(*) AS complaints FROM complaints GROUP BY 1,2,3 ORDER BY 1,2,3""", con)\n        sql_file=root/"sql/business_analysis.sql"\n        if sql_file.exists():\n            statements=[s.strip() for s in sql_file.read_text(encoding="utf-8").split(";") if s.strip()]\n            for number,statement in enumerate(statements,1):\n                pd.read_sql_query(statement,con).to_csv(output/f"sql_{number:02d}.csv",index=False)\n    weekly.to_csv(output / "weekly_issues.csv", index=False)\n    analytics["signals"] = detect_signals(weekly, "2025-01-01", "2025-02-28")\n    complete_weeks=sorted(w for w in weekly["week"].unique() if w>="2025-01-01" and pd.Timestamp(w)+pd.Timedelta(days=6)<=pd.Timestamp("2025-02-28"))\n    analytics["signal_history"]=[detect_signals(weekly,"2025-01-01",(pd.Timestamp(week)+pd.Timedelta(days=6)).strftime("%Y-%m-%d")) for week in complete_weeks[4:]]\n    write_json(output / "signals.json", analytics["signals"])\n    write_json(output / "signal_history.json", analytics["signal_history"])\n    return analytics\n\n\ndef detect_signals(weekly: pd.DataFrame, date_min: str, date_max: str) -> dict:\n    """Compare last complete week against four complete prior weeks; partial weeks excluded."""\n    first, last = pd.Timestamp(date_min), pd.Timestamp(date_max)\n    complete = sorted(w for w in weekly["week"].unique() if pd.Timestamp(w) >= first and pd.Timestamp(w) + pd.Timedelta(days=6) <= last)\n    if len(complete) < 5:\n        return {"status": "insufficient_history", "items": []}\n    current, baseline = complete[-1], complete[-5:-1]\n    totals = weekly.groupby("week")["complaints"].sum()\n    current_total = int(totals[current])\n    baseline_total = int(totals.loc[baseline].sum())\n    recent = weekly[weekly.week == current].groupby(["product", "issue"])["complaints"].sum()\n    previous = weekly[weekly.week.isin(baseline)].groupby(["product", "issue"])["complaints"].sum()\n    items = []\n    for (product, issue), count in recent.items():\n        before = int(previous.get((product, issue), 0))\n        average = before / 4\n        ratio = float(count / average) if average else None\n        delta_pp = 100 * (count / current_total - before / baseline_total)\n        if count >= 30 and average >= 5 and ratio >= 1.5 and delta_pp >= 0.5:\n            items.append({"product": product, "issue": issue, "current": int(count), "baseline_weekly_average": average,\n                          "volume_ratio": round(ratio, 3), "share_change_pp": round(delta_pp, 3)})\n    return {"status": "descriptive_heuristic", "week_start": current, "baseline_weeks": baseline,\n            "current_week_total": current_total, "baseline_four_week_total": baseline_total,\n            "rule": "count >=30, baseline weekly mean >=5, volume >=1.5x baseline, share increase >=0.5 percentage points",\n            "limitations": "Exploratory signals, not statistically confirmed anomalies or proof of company misconduct. Only seven complete weeks are available.",\n            "items": sorted(items, key=lambda x: -x["share_change_pp"])}\n\n\ndef load_model_data(root: Path):\n    with closing(sqlite3.connect(root / "data/complaints.sqlite")) as con:\n        frame = pd.read_sql_query("""SELECT complaint_id,date_received,product,clean_text,text_hash FROM\n          (SELECT *,ROW_NUMBER() OVER(PARTITION BY text_hash ORDER BY date_received,complaint_id) AS rn\n           FROM complaints WHERE clean_text IS NOT NULL) WHERE rn=1 ORDER BY date_received,complaint_id""", con)\n        eligible_before = con.execute("SELECT COUNT(*) FROM complaints WHERE clean_text IS NOT NULL").fetchone()[0]\n    return frame, {"eligible_before_dedup": eligible_before, "normalized_duplicates_removed": eligible_before-len(frame), "unique_eligible_narratives": len(frame)}\n\n\ndef cap_classes(frame: pd.DataFrame, cap: int | None):\n    if cap is None:\n        return frame.copy()\n    selected = []\n    for _, group in frame.groupby("product", sort=True):\n        selected.append(group.sample(n=min(cap, len(group)), random_state=SEED))\n    return pd.concat(selected).sort_values(["date_received", "complaint_id"]).reset_index(drop=True)\n\n\ndef membership_fingerprint(frame: pd.DataFrame) -> str:\n    columns=["complaint_id","date_received","product","text_hash"]\n    rows=frame[columns].astype(str).sort_values("complaint_id").values.tolist()\n    return hashlib.sha256(json.dumps(rows,ensure_ascii=False,separators=(",",":")).encode()).hexdigest()\n\n\ndef make_splits(frame: pd.DataFrame, profile: str):\n    cfg = PROFILES[profile]\n    windows = {\n        "train": frame[frame.date_received <= "2025-02-09"],\n        "validation": frame[(frame.date_received >= "2025-02-10") & (frame.date_received <= "2025-02-16")],\n        "test": frame[frame.date_received >= "2025-02-17"],\n    }\n    labels = sorted(windows["train"][\'product\'].value_counts().loc[lambda x: x >= 20].index.tolist())\n    if len(labels) < 2:\n        raise ValueError("Insufficient labels in training window.")\n    splits = {name: cap_classes(part[part[\'product\'].isin(labels)], cfg["train_cap_per_class"] if name=="train" else cfg["eval_cap_per_class"]) for name,part in windows.items()}\n    for name, part in splits.items():\n        if part.empty:\n            raise ValueError(f"Empty {name} split.")\n    sets = {name: set(part.text_hash) for name,part in splits.items()}\n    assert not (sets["train"] & sets["validation"] or sets["train"] & sets["test"] or sets["validation"] & sets["test"]), "Duplicate leakage"\n    manifest = {\n        "strategy": "Chronological; keep earliest normalized text once before splitting. Vocabulary is fitted only on training text.",\n        "profile": profile, "seed": SEED, "train_cap_per_class": cfg["train_cap_per_class"],\n        "training_rows_before_cap": int(windows["train"][\'product\'].isin(labels).sum()),\n        "training_rows_excluded_by_cap": int(windows["train"][\'product\'].isin(labels).sum())-len(splits["train"]),\n        "evaluation_capped": cfg["eval_cap_per_class"] is not None,\n        "evaluation_scope": ("Category-capped pipeline smoke check; not a population performance estimate." if profile=="smoke" else\n                             "All retained, in-scope unique narratives in each chronological window; all eligible training-window narratives are used." if profile=="uncapped" else\n                             "All retained, in-scope unique narratives in each chronological evaluation window; training is capped by category."),\n        "labels": labels, "out_of_scope_rows": int((~frame[\'product\'].isin(labels)).sum()),\n        "normalized_hash_overlap": 0,\n        "holdout_fingerprints": {name:membership_fingerprint(splits[name]) for name in ("validation","test")},\n        "near_duplicate_limit": "Normalization catches exact and variable-identifier templates. Paraphrases and other near-duplicates may remain.",\n        "splits": {name: {"rows": len(part), "date_min": part.date_received.min(), "date_max": part.date_received.max(),\n                           "class_counts": {str(k):int(v) for k,v in part[\'product\'].value_counts().items()}} for name,part in splits.items()},\n    }\n    return splits, manifest\n\n\ndef load_capped_reference(root: Path, splits: dict):\n    path=root/"reference/capped_full_reference.json"\n    if not path.exists():\n        raise ValueError("The uncapped comparison needs reference/capped_full_reference.json from the supplied notebook or project package.")\n    reference=json.loads(path.read_text(encoding="utf-8"))\n    if reference["source_sha256"]!=SOURCE_SHA256:\n        raise ValueError("Capped reference uses a different dataset.")\n    for name in ("validation","test"):\n        expected=reference["holdout_membership"][name]\n        if len(splits[name])!=expected["rows"] or membership_fingerprint(splits[name])!=expected["fingerprint"]:\n            raise ValueError(f"The {name} complaints differ from the capped run. Do not compare scores from different holdouts.")\n    return reference\n\n\ndef compare_validation_to_capped(reference: dict, selected: dict):\n    baseline=float(reference["selected_validation_macro_f1"])\n    score=float(selected["macro_f1"])\n    recommend="uncapped" if score>baseline+1e-12 else "full"\n    return {"criterion":"validation macro-F1 only; test metrics do not choose the profile",\n            "capped_selected_model":reference["selected_model"],"capped_training_rows":reference["training_rows"],\n            "capped_validation_macro_f1":baseline,"uncapped_selected_model":selected["model"],\n            "uncapped_validation_macro_f1":score,"delta_validation_macro_f1":score-baseline,\n            "recommended_profile":recommend,"tie_rule":"Preserve capped model when validation macro-F1 ties.",\n            "reference_archive_sha256":reference["reference_archive_sha256"],"identical_holdouts_verified":True}\n\n\ndef metrics(y, pred, labels):\n    return {"accuracy": float(accuracy_score(y,pred)), "balanced_accuracy": float(balanced_accuracy_score(y,pred)),\n            "macro_f1": float(f1_score(y,pred,labels=labels,average="macro",zero_division=0)),\n            "weighted_f1": float(f1_score(y,pred,labels=labels,average="weighted",zero_division=0))}\n\n\ndef fit_models(root: Path, splits: dict, split_manifest: dict, profile: str, execution_environment: str):\n    cfg = PROFILES[profile]\n    train, valid, test = (splits[x] for x in ("train", "validation", "test"))\n    labels = split_manifest["labels"]\n    reference=load_capped_reference(root,splits) if profile=="uncapped" else None\n    models = {\n        "majority_baseline": DummyClassifier(strategy="most_frequent"),\n        "complement_naive_bayes": ComplementNB(alpha=0.5),\n        "logistic_regression": LogisticRegression(C=2.0, class_weight="balanced", max_iter=350, solver="lbfgs", random_state=SEED),\n    }\n    candidates, fitted = [], {}\n    for name, estimator in models.items():\n        start = time.perf_counter()\n        pipeline = Pipeline([("tfidf", TfidfVectorizer(preprocessor=normalize_text, ngram_range=(1,2), min_df=2,\n                                                     max_df=0.98, max_features=cfg["max_features"], sublinear_tf=True, dtype=np.float32)),\n                             ("classifier", estimator)])\n        pipeline.fit(train.clean_text.tolist(), train[\'product\'].tolist())\n        prediction = pipeline.predict(valid.clean_text.tolist())\n        row = {"model": name, **metrics(valid[\'product\'],prediction,labels), "fit_and_validation_seconds": round(time.perf_counter()-start,2)}\n        candidates.append(row)\n        fitted[name] = pipeline\n        print(f"Validation: {name}: macro-F1={row[\'macro_f1\']:.4f}",flush=True)\n    winner_row = max((row for row in candidates if row["model"]!="majority_baseline"),key=lambda row:row["macro_f1"])\n    winner = winner_row["model"]\n    comparison=compare_validation_to_capped(reference,winner_row) if reference else None\n    model = fitted[winner]\n    probabilities = model.predict_proba(valid.clean_text.tolist())\n    confidence = probabilities.max(axis=1)\n    validation_pred = model.classes_[probabilities.argmax(axis=1)]\n    threshold_rows=[]\n    for threshold in np.arange(0.4,0.951,0.05):\n        keep = confidence >= threshold\n        threshold_rows.append({"threshold": round(float(threshold),2), "coverage": float(keep.mean()), "accepted": int(keep.sum()),\n                               "accuracy_when_accepted": float(np.mean(validation_pred[keep]==valid[\'product\'].to_numpy()[keep])) if keep.any() else None})\n    acceptable = [row for row in threshold_rows if row["accepted"]>=20 and row["coverage"]>=0.1 and row["accuracy_when_accepted"]>=0.85]\n    threshold = max(acceptable,key=lambda row:row["coverage"])["threshold"] if acceptable else 0.9\n    test_pred = model.predict(test.clean_text.tolist())\n    test_proba = model.predict_proba(test.clean_text.tolist())\n    test_scores = test_proba.max(axis=1)\n    accepted = test_scores >= threshold\n    report = classification_report(test[\'product\'],test_pred,labels=labels,output_dict=True,zero_division=0)\n    confusion = confusion_matrix(test[\'product\'],test_pred,labels=labels)\n    results = {\n        "pipeline_version": VERSION, "profile": profile, "execution_environment": execution_environment,\n        "execution_time_utc": datetime.now(timezone.utc).isoformat(),\n        "python_version": platform.python_version(), "sklearn_version": sklearn.__version__,\n        "pandas_version": pd.__version__, "numpy_version": np.__version__,\n        "selected_model": winner, "selection_metric": "validation macro-F1; test used only after model and routing-threshold selection",\n        "training_cap_per_class":cfg["train_cap_per_class"],\n        "comparison_to_capped":comparison,\n        "validation_models": candidates, "test": metrics(test[\'product\'],test_pred,labels),\n        "test_baseline": metrics(test[\'product\'],fitted["majority_baseline"].predict(test.clean_text.tolist()),labels),\n        "classification_report": report, "labels": labels, "confusion_matrix": confusion.tolist(),\n        "routing": {"threshold": threshold, "validation_target_accuracy":0.85, "validation_target_met": bool(acceptable),\n                    "test_coverage": float(accepted.mean()), "test_accepted": int(accepted.sum()),\n                    "test_accuracy_when_accepted": float(np.mean(test_pred[accepted]==test[\'product\'].to_numpy()[accepted])) if accepted.any() else None,\n                    "score_note": "Uncalibrated model score; not a guaranteed probability. Review route is a demonstration, not autonomous financial decision-making."},\n        "threshold_search": threshold_rows,\n        "evidence_status": (f"Smoke profile executed in {execution_environment}; small pipeline check, not final population performance." if profile=="smoke" else\n                            f"{profile.title()} profile executed in {execution_environment}; interpret within documented dataset and split scope."),\n    }\n    models_dir = root/"models"/profile\n    reports_dir = root/"reports"/profile\n    models_dir.mkdir(parents=True,exist_ok=True)\n    reports_dir.mkdir(parents=True,exist_ok=True)\n    joblib.dump(model,models_dir/"classifier.joblib",compress=3)\n    write_json(models_dir/"routing.json",results["routing"])\n    write_json(reports_dir/"metrics.json",results)\n    write_json(reports_dir/"split_manifest.json",split_manifest)\n    if reference:\n        write_json(reports_dir/"capped_reference.json",reference)\n        write_json(reports_dir/"comparison_to_capped.json",comparison)\n    pd.DataFrame(candidates).to_csv(reports_dir/"validation_models.csv",index=False)\n    pd.DataFrame(report).transpose().to_csv(reports_dir/"classification_report.csv")\n    pd.DataFrame(confusion,index=labels,columns=labels).to_csv(reports_dir/"confusion_matrix.csv")\n    for name,part in splits.items():\n        part[["complaint_id","date_received","product","text_hash"]].to_csv(reports_dir/f"{name}_membership.csv",index=False)\n    pd.DataFrame({"complaint_id":test.complaint_id,"actual":test[\'product\'],"predicted":test_pred,\n                  "score":test_scores,"human_review":~accepted}).to_csv(reports_dir/"test_predictions.csv",index=False)\n    # Verify the portable artifact immediately after writing.\n    loaded = joblib.load(models_dir/"classifier.joblib")\n    assert np.array_equal(loaded.predict(test.clean_text.iloc[:10].tolist()),test_pred[:10])\n    write_json(models_dir/"artifact_manifest.json",{"model_sha256":file_sha(models_dir/"classifier.joblib"),\n               "profile":profile,"execution_environment":execution_environment,"sklearn_version":sklearn.__version__,\n               "python_version":platform.python_version(),"warning":"Only load joblib files you generated or trust."})\n    return results\n\n\ndef publish_dashboard_data(root: Path, analytics: dict, results: dict, manifest: dict):\n    data = {"source":manifest,"analytics":analytics,"model":results,\n            "project":"Financial Complaint Intelligence","historical":True}\n    write_json(root/"reports"/results["profile"]/"dashboard_data.json",data)\n\n\ndef make_figures(root: Path, results: dict):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    folder=root/"reports"/results["profile"]\n    plt.rcParams.update({"font.family":"DejaVu Sans","font.size":10,"axes.spines.top":False,"axes.spines.right":False})\n    rows=results["validation_models"]\n    fig,ax=plt.subplots(figsize=(8,4.8),layout="constrained")\n    names=[row["model"].replace("_"," ").title() for row in rows]\n    bars=ax.barh(names,[row["macro_f1"] for row in rows],color=["#b0b8c5","#5b9c9a","#123b56"])\n    ax.set(xlim=(0,1),xlabel="Validation macro-F1",title=f"Model comparison | {results[\'profile\']} profile")\n    for bar,row in zip(bars,rows):\n        ax.text(min(row["macro_f1"]+0.02,0.95),bar.get_y()+bar.get_height()/2,f"{row[\'macro_f1\']:.3f}",va="center")\n    fig.savefig(folder/"validation_comparison.png",dpi=180)\n    plt.close(fig)\n    matrix=np.asarray(results["confusion_matrix"],dtype=float)\n    normalized=matrix/np.maximum(matrix.sum(axis=1,keepdims=True),1)\n    labels=[x[:28]+("..." if len(x)>28 else "") for x in results["labels"]]\n    fig,ax=plt.subplots(figsize=(11,9),layout="constrained")\n    plotted=ax.imshow(normalized,vmin=0,vmax=1,cmap="Blues")\n    ax.set(xticks=range(len(labels)),yticks=range(len(labels)),xticklabels=labels,yticklabels=labels,\n           xlabel="Predicted product",ylabel="Actual product",title=f"Held-out test confusion matrix | {results[\'profile\']} profile\\nRows normalized; this does not measure business impact")\n    plt.setp(ax.get_xticklabels(),rotation=65,ha="right",rotation_mode="anchor",fontsize=8)\n    for i in range(len(labels)):\n        for j in range(len(labels)):\n            if normalized[i,j]>=0.05:\n                ax.text(j,i,f"{normalized[i,j]:.2f}",ha="center",va="center",fontsize=8,color="white" if normalized[i,j]>0.55 else "#10243a")\n    fig.colorbar(plotted,ax=ax,shrink=0.7,label="Fraction of actual category")\n    fig.savefig(folder/"confusion_matrix.png",dpi=170)\n    plt.close(fig)\n\n\ndef export_bundle(root: Path, profile: str) -> Path:\n    target=root/f"Financial_Complaint_Results_{profile}.zip"\n    with zipfile.ZipFile(target,"w",zipfile.ZIP_DEFLATED) as archive:\n        for folder in [root/"reports"/profile,root/"reports/analytics",root/"models"/profile]:\n            for path in sorted(folder.rglob("*")):\n                if path.is_file():\n                    archive.write(path,path.relative_to(root).as_posix())\n        archive.write(root/"reports/data_manifest.json","reports/data_manifest.json")\n        archive.writestr("EXECUTION_STATUS.txt",f"Profile: {profile}\\nRead reports/{profile}/metrics.json for the actual execution environment and results.\\nThis results archive does not include the raw archive or SQLite database.\\n")\n    return target\n\n\ndef run(root: Path, profile: str="smoke", execution_environment: str="local_cpu"):\n    root=root.resolve()\n    print(f"Starting {profile} profile in {execution_environment}",flush=True)\n    archive=download_archive(root)\n    manifest=build_database(root,archive)\n    analytics=export_analytics(root)\n    frame,quality=load_model_data(root)\n    splits,split_manifest=make_splits(frame,profile)\n    split_manifest.update(quality)\n    print({name:len(part) for name,part in splits.items()},flush=True)\n    results=fit_models(root,splits,split_manifest,profile,execution_environment)\n    make_figures(root,results)\n    publish_dashboard_data(root,analytics,results,manifest)\n    bundle=export_bundle(root,profile)\n    print(f"Finished. Results: {bundle}",flush=True)\n    return results\n\n\nif __name__=="__main__":\n    # Keep pickled preprocessing functions importable under the same module name.\n    import complaint_project\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--profile",choices=PROFILES,default="smoke")\n    parser.add_argument("--root",type=Path,default=Path(__file__).resolve().parent)\n    parser.add_argument("--environment",default="local_cpu")\n    args=parser.parse_args()\n    complaint_project.run(args.root,args.profile,args.environment)\n'
SQL_SOURCE = "-- All-record reporting population: January-February 2025 archive.\n-- No narrative-text filter is applied to these business metrics.\n\n-- 1. Monthly product volume and change, using a window function.\nWITH monthly AS (\n    SELECT substr(date_received,1,7) AS month,product,COUNT(*) AS complaints\n    FROM complaints GROUP BY 1,2\n), previous AS (\n    SELECT *,LAG(complaints) OVER(PARTITION BY product ORDER BY month) AS prior_month\n    FROM monthly\n)\nSELECT *,ROUND(100.0*(complaints-prior_month)/NULLIF(prior_month,0),2) AS change_pct\nFROM previous ORDER BY product,month;\n\n-- 2. Top three reported issues in each product category.\nWITH counts AS (\n    SELECT product,issue,COUNT(*) AS complaints FROM complaints GROUP BY 1,2\n), ranked AS (\n    SELECT *,DENSE_RANK() OVER(PARTITION BY product ORDER BY complaints DESC) AS issue_rank\n    FROM counts\n)\nSELECT * FROM ranked WHERE issue_rank<=3 ORDER BY product,issue_rank;\n\n-- 3. Narrative coverage. The modeling subset differs from all-record analytics.\nSELECT product,COUNT(*) AS complaints,SUM(has_narrative) AS narratives,\n       ROUND(100.0*SUM(has_narrative)/COUNT(*),2) AS narrative_coverage_pct\nFROM complaints GROUP BY product ORDER BY complaints DESC;\n\n-- 4. Timely-response percentage with an explicit known-value denominator.\n-- This is a source-provided binary flag, not a measured response duration.\nSELECT product,\n       SUM(CASE WHEN timely_response='Yes' THEN 1 ELSE 0 END) AS timely,\n       SUM(CASE WHEN timely_response IN ('Yes','No') THEN 1 ELSE 0 END) AS known,\n       ROUND(100.0*SUM(CASE WHEN timely_response='Yes' THEN 1 ELSE 0 END)/\n             NULLIF(SUM(CASE WHEN timely_response IN ('Yes','No') THEN 1 ELSE 0 END),0),2) AS timely_pct\nFROM complaints GROUP BY product;\n\n-- 5. Product share by month: join group counts to their monthly totals.\nWITH product_counts AS (\n    SELECT substr(date_received,1,7) AS month,product,COUNT(*) AS complaints\n    FROM complaints GROUP BY 1,2\n), month_counts AS (\n    SELECT substr(date_received,1,7) AS month,COUNT(*) AS total FROM complaints GROUP BY 1\n)\nSELECT p.*,m.total,ROUND(100.0*p.complaints/m.total,2) AS share_pct\nFROM product_counts p JOIN month_counts m ON p.month=m.month\nORDER BY p.month,p.complaints DESC;\n\n-- 6. Check that normalized text templates do not cross model partitions.\n-- These fixed windows match complaint_project.make_splits.\nWITH membership AS (\n SELECT text_hash,\n CASE WHEN date_received<='2025-02-09' THEN 'train'\n      WHEN date_received<='2025-02-16' THEN 'validation' ELSE 'test' END AS partition_name,\n ROW_NUMBER() OVER(PARTITION BY text_hash ORDER BY date_received,complaint_id) AS rn\n FROM complaints WHERE clean_text IS NOT NULL\n)\nSELECT text_hash,COUNT(DISTINCT partition_name) AS partitions\nFROM membership WHERE rn=1 GROUP BY text_hash HAVING COUNT(DISTINCT partition_name)>1;\n"
CAPPED_REFERENCE_SOURCE = '{\n  "source_sha256": "11c18bb0973eed8f5e4f66e2979346af0671a3aefcb5a6b9eec73009d53cff64",\n  "reference_archive_sha256": "f29504445d3d7754ce4ed2850bd22f26c5066c8844663c4eb9781f1d1381e359",\n  "execution_environment": "colab_cpu",\n  "selected_model": "logistic_regression",\n  "selected_validation_macro_f1": 0.6538825375829431,\n  "selected_validation_accuracy": 0.7783444904898523,\n  "training_rows": 60444,\n  "train_cap_per_class": 15000,\n  "test": {\n    "accuracy": 0.7842105263157895,\n    "balanced_accuracy": 0.7223780261934952,\n    "macro_f1": 0.6579507178401467,\n    "weighted_f1": 0.79346057280516\n  },\n  "holdout_membership": {\n    "validation": {\n      "rows": 9411,\n      "fingerprint": "0c1cf57bd276e52752ebed1cbd2d521a64ad360b77712a6a18c04f9edf66061a"\n    },\n    "test": {\n      "rows": 15390,\n      "fingerprint": "a00bd0d820045066a5ac98627b6a231060294c7ab4eb924b261481e432b1f328"\n    }\n  }\n}\n'
(ROOT / "complaint_project.py").write_text(PIPELINE_SOURCE, encoding="utf-8")
(ROOT / "sql").mkdir(exist_ok=True)
(ROOT / "sql/business_analysis.sql").write_text(SQL_SOURCE, encoding="utf-8")
(ROOT / "reference").mkdir(exist_ok=True)
(ROOT / "reference/capped_full_reference.json").write_text(CAPPED_REFERENCE_SOURCE, encoding="utf-8")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_setup import assert_no_pending_restart
assert_no_pending_restart(ROOT)
import complaint_project as project
import importlib
importlib.reload(project)
import pandas as pd
import numpy as np
import scipy, sklearn, joblib
for package, module in {"numpy": np, "pandas": pd, "scipy": scipy, "scikit-learn": sklearn, "joblib": joblib}.items():
    assert module.__version__ == REQUIRED[package], f"Restart the runtime and rerun: {package} is already imported with a different version."
from IPython.display import display, Image, FileLink
print("Pipeline version:", project.VERSION)

## 4. Download and verify the real data

Source: [CFPB official narratives archive](https://www.consumerfinance.gov/foia-requests/foia-electronic-reading-room/cfpb-consumer-complaint-database-narratives-archive/), January-February 2025.

The current main complaint export no longer includes narrative text, so this project intentionally uses the historical archive. A fixed SHA-256 checksum prevents silently changing the source dataset. If the publisher changes its archive, inspect the new version before updating that checksum.

In [ ]:
archive_path = project.download_archive(ROOT)
assert project.file_sha(archive_path) == project.SOURCE_SHA256
print("Verified archive:", archive_path.name)
print("SHA-256:", project.SOURCE_SHA256)

## 5. Clean, validate and store the data in SQL

The original CSV is streamed into SQLite. All archived records feed the business analysis; only sufficiently long written narratives feed the classifier. URLs, email addresses, digits and redaction markers are removed from modeling text.

Complaint descriptions are unverified consumer accounts. The data is self-selected and is not representative of all customers. Complaint counts do not establish company misconduct or customer failure rates.

In [ ]:
data_manifest = project.build_database(ROOT, archive_path)
display(pd.Series(data_manifest, name="source_and_quality"))
assert data_manifest["source_rows"] == 811019
assert data_manifest["with_narrative"] == 265405
assert data_manifest["database_rows"] == data_manifest["source_rows"] - data_manifest["duplicate_ids_removed"]

## 6. Business analysis and SQL evidence

Analytics include records without narrative text. The provided SQL file includes joins, common table expressions, ranking, monthly comparisons, narrative coverage and response metrics.

`Timely response?` is a source-provided yes/no field. It is not a measured response duration.

In [ ]:
analytics = project.export_analytics(ROOT)
display(pd.DataFrame(analytics["products"]))
display(pd.DataFrame(analytics["issues"]).head(15))

from contextlib import closing
import sqlite3
with closing(sqlite3.connect(ROOT / "data/complaints.sqlite")) as connection:
    top_issues = pd.read_sql_query("""
    WITH counts AS (
      SELECT product, issue, COUNT(*) AS complaints FROM complaints GROUP BY 1,2
    ), ranked AS (
      SELECT *, DENSE_RANK() OVER(PARTITION BY product ORDER BY complaints DESC) AS issue_rank FROM counts
    ) SELECT * FROM ranked WHERE issue_rank <= 3 ORDER BY product,issue_rank
    """, connection)
display(top_issues)
top_issues.to_csv(ROOT / "reports/analytics/top_issues_per_product.csv", index=False)

## 7. Remove repeated text and define chronological partitions

Keep the earliest occurrence of each normalized narrative. This excludes identical text and templates that differ only in removed identifiers. Other near-duplicates and paraphrases may remain; we do not claim they are all removed.

- **Train:** January 1-February 9, 2025.
- **Validation:** February 10-16, 2025.
- **Test:** February 17-28, 2025.

Categories need at least 20 unique training examples. The **uncapped** profile uses every eligible training-window complaint, with class weighting retained in Logistic Regression. The old `full` profile retains its 15,000-per-category training cap for reproducibility. Smoke evaluation is capped by category for pipeline checking. Both `full` and `uncapped` retain all eligible, deduplicated, in-scope records in the same validation/test windows.

In [ ]:
model_frame, duplicate_quality = project.load_model_data(ROOT)
splits, split_manifest = project.make_splits(model_frame, PROFILE)
split_manifest.update(duplicate_quality)
display(pd.Series(duplicate_quality))
display(pd.DataFrame({name: part["product"].value_counts() for name, part in splits.items()}).fillna(0).astype(int))
print("Normalized text overlap:", split_manifest["normalized_hash_overlap"])
print(split_manifest["evaluation_scope"])
assert split_manifest["normalized_hash_overlap"] == 0
print("Eligible training rows before cap:", split_manifest["training_rows_before_cap"])
print("Training rows excluded by cap:", split_manifest["training_rows_excluded_by_cap"])
if PROFILE == "uncapped":
    assert {name: len(part) for name, part in splits.items()} == {"train": 101935, "validation": 9411, "test": 15390}
    assert split_manifest["training_rows_excluded_by_cap"] == 0
    capped_reference = project.load_capped_reference(ROOT, splits)
    print("Confirmed: identical validation/test membership to the preserved capped Colab run.")

## 8. Fit a baseline and two text models

**TF-IDF** turns words and pairs of words into numerical features. Frequent but less informative words receive less weight. Its vocabulary is fitted only on training text.

Compare a majority-class baseline, Complement Naive Bayes, and Logistic Regression. Select the useful model with the highest **validation macro-F1**, which gives each category equal importance. The test set does not choose the model.

A review threshold is chosen using validation data. The resulting model scores are **uncalibrated**, so they should not be described as guaranteed probabilities.

For the uncapped experiment, the selected model is also compared with your preserved capped model using validation macro-F1. The capped reference is embedded in this notebook, including hashes of the exact holdout membership. Test scores do not determine the recommendation. A tie keeps the earlier capped model.

In [ ]:
results = project.fit_models(ROOT, splits, split_manifest, PROFILE, EXECUTION_ENVIRONMENT)
display(pd.DataFrame(results["validation_models"]))
print("Selected model:", results["selected_model"])
print("Selection:", results["selection_metric"])
if results.get("comparison_to_capped"):
    display(pd.Series(results["comparison_to_capped"], name="capped_vs_uncapped_validation_comparison"))

## 9. Evaluate the held-out test period

Macro-F1 measures performance across categories; weighted F1 and accuracy reflect class frequencies. Read per-category precision and recall as well as the aggregate score.

The score is conditional on this historical, narrative-only, deduplicated dataset. It does not measure customer satisfaction, causal business impact, or performance on arbitrary financial messages.

In [ ]:
display(pd.Series(results["test"], name="test_metrics"))
display(pd.DataFrame(results["classification_report"]).T)
display(pd.Series(results["routing"], name="human_review_routing"))
print(results["evidence_status"])
project.make_figures(ROOT, results)
display(Image(filename=str(ROOT / "reports" / PROFILE / "validation_comparison.png")))
display(Image(filename=str(ROOT / "reports" / PROFILE / "confusion_matrix.png")))

## 10. Emerging issue signals

Compare the last complete week with four preceding complete weeks. Ignore partial boundary weeks. Require at least 30 complaints, a baseline weekly mean of at least 5, at least 1.5 times the baseline volume, and a share increase of at least 0.5 percentage points.

These are transparent exploratory rules, not statistically confirmed anomalies. With only two months of data, long-term seasonality cannot be evaluated. An empty result means no issue met the stated thresholds; it is not an error.

In [ ]:
signals = analytics["signals"]
print("Latest complete week:", signals.get("week_start"))
print("Baseline weeks:", signals.get("baseline_weeks"))
display(pd.DataFrame(signals["items"]) if signals["items"] else "No issue met all signal thresholds.")
historical_signals = [{"week_start": window["week_start"], **item} for window in analytics["signal_history"] for item in window["items"]]
print("Historical comparison windows:", len(analytics["signal_history"]))
display(pd.DataFrame(historical_signals) if historical_signals else "No historical window met all signal thresholds.")
print(signals.get("limitations", ""))

## 11. Try the exported classifier

The following is an authored demonstration input, not a labeled test example and not part of the reported evaluation. The classifier suggests one of its trained product categories; it does not provide financial advice or verify allegations.

In [ ]:
import joblib
model = joblib.load(ROOT / "models" / PROFILE / "classifier.joblib")
EXAMPLE_COMPLAINT = "My credit card shows a purchase that I did not authorize and the card issuer has refused to investigate my dispute."
scores = model.predict_proba([EXAMPLE_COMPLAINT])[0]
ranked = np.argsort(scores)[::-1][:3]
display(pd.DataFrame({"suggested_product": model.classes_[ranked], "uncalibrated_score": scores[ranked]}))
print("Human review:", bool(scores.max() < results["routing"]["threshold"]))

## 12. Export results for the local dashboard

The ZIP includes the fitted pipeline, checksum, model/environment details, split membership, test predictions, figures, SQL analytics and dashboard data. It excludes the raw data archive and SQLite database.

Only load model files you generated or trust. The local application must use the same scikit-learn version as the exported model.

In [ ]:
project.publish_dashboard_data(ROOT, analytics, results, data_manifest)
bundle = project.export_bundle(ROOT, PROFILE)
if SAVE_RESULTS_TO_DRIVE:
    assert IN_COLAB, "Drive saving is only supported by this cell in Colab."
    from google.colab import drive
    drive.mount("/content/drive")
    drive_folder = Path("/content/drive/MyDrive/FinancialComplaintIntelligence")
    drive_folder.mkdir(parents=True, exist_ok=True)
    shutil.copy2(bundle, drive_folder / bundle.name)
    print("Saved results to:", drive_folder / bundle.name)
print("Results ZIP:", bundle)
print("Execution environment:", results["execution_environment"])
print("Profile:", PROFILE)
if PROFILE == "smoke":
    print('NEXT: change PROFILE to "uncapped" and run all cells again. Smoke scores are not final project performance.')
else:
    print("NEXT: download", bundle.name, "and save the executed notebook. Return both for verification.")
    if PROFILE == "uncapped":
        print("Keep your original Financial_Complaint_Results_full.zip as the capped reference.")

## 13. Download and retain your evidence

Run this cell after the uncapped profile completes. **Financial_Complaint_Results_uncapped.zip** and the executed notebook are the handoff files. The ZIP alone does not contain the notebook. The earlier capped result has a different filename and is preserved.

In Colab, use **File > Download > Download .ipynb** to save this notebook with its cell outputs.

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))
else:
    display(FileLink(str(bundle)))

## What to explain in an interview

1. Why narrative coverage differs from all-record complaint analysis.
2. Why repeated templates and future information can inflate model scores.
3. Why chronological validation and test windows were used.
4. Why macro-F1 and per-category recall matter for imbalanced data.
5. How the review threshold trades automation coverage against errors.
6. Why a complaint-volume signal is not proof of misconduct or a causal business result.
7. What changes before production: broader time coverage, stronger near-duplicate checks, external validation, score calibration, privacy review and monitoring.

## Completion boundary

The uncapped experiment is complete when its Colab run and exported artifacts have been verified against the capped reference. More training data does not guarantee better validation performance; preserve the validation-selected model. The professional dashboard implementation follows the user's Superdesign design review. Do not list Power BI, Databricks, deployment, business savings, or uncapped-model results unless those steps were actually completed and verified.